In [0]:
%run /Workspace/Users/sushantbhardwaj15@gmail.com/FMCG_Analytics/setup_utils_config/Utilities

In [0]:
from pyspark.sql import functions as F

for table_name, cfg in TABLE_CONFIG.items():
    if cfg["load_type"] != "full":
        continue  # skip orders — handled by autoloader

    print(f"\n[BRONZE LAYER] Starting ingestion for table: {table_name}")

    try:
        # 1. Read from S3
        df = (
            spark.read
            .format(cfg["format"])
            .option("header", True)
            .option("inferSchema", True)
            .load(cfg["source"])
        )

        # 2. Add audit columns
        df = (
            df
            .withColumn("_source_file", F.col("_metadata.file_name"))
            .withColumn("_ingested_at", F.current_timestamp())
            .withColumn("_load_date", F.current_date())
        )
        (
            df.write
            .format("delta")
            .option("enableChangeDataFeed", "true")
            .mode("overwrite")
            .option("overwriteSchema", True)
            .saveAsTable(f"{catalog}.{bronze_schema}.{table_name}")
        )

        # 4. Write audit log
        write_audit("bronze", table_name, df.count(), "SUCCESS")

    except Exception as e:
        print(f"ERROR: {e}")
        write_audit("bronze", table_name, 0, "FAILED", str(e))